Question 1 : Use Github Copilot to create a table SalesData whose columns are CustomerID, Name, Age, City,
PurchaseAmount, PurchaseDate and then ask to insert 10,000 rows of random data.

  -In **SSMS with GitHub Copilot enabled**, you can give Copilot a prompt like this and let it generate the SQL.

```sql
-- Create a table named SalesData with:
-- CustomerID (Primary Key, Auto Increment)
-- Name
-- Age
-- City
-- PurchaseAmount
-- PurchaseDate
-- Then insert 10,000 rows of random sample data

CREATE TABLE SalesData (
    CustomerID INT IDENTITY(1,1) PRIMARY KEY,
    Name VARCHAR(100),
    Age INT,
    City VARCHAR(100),
    PurchaseAmount DECIMAL(10,2),
    PurchaseDate DATE
);

-- Insert 10,000 random rows
WITH Numbers AS (
    SELECT TOP (10000)
        ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS Num
    FROM sys.objects a
    CROSS JOIN sys.objects b
)

INSERT INTO SalesData (Name, Age, City, PurchaseAmount, PurchaseDate)
SELECT
    CONCAT('Customer_', Num),
    FLOOR(RAND(CHECKSUM(NEWID())) * 43) + 18,   -- Age 18–60
    CHOOSE(
        FLOOR(RAND(CHECKSUM(NEWID())) * 5) + 1,
        'Delhi',
        'Mumbai',
        'Bangalore',
        'Chennai',
        'Kolkata'
    ),
    ROUND((RAND(CHECKSUM(NEWID())) * 5000) + 100, 2),
    DATEADD(
        DAY,
        -FLOOR(RAND(CHECKSUM(NEWID())) * 365),
        GETDATE()
    )
FROM Numbers;

-- Verify data
SELECT TOP 10 * FROM SalesData;

-- Check row count
SELECT COUNT(*) AS TotalRows
FROM SalesData;
```

**How to use Copilot in SSMS:**

1. Open a new SQL query window.
2. Paste the comments (instructions) or type the requirement.
3. Press **Tab** or wait for **GitHub Copilot suggestions**.
4. Accept the generated SQL and run it.

This creates **SalesData** and inserts **10,000 random records**.

Question 2 : Use Copilot to:

Find total sales per city

Top 5 cities by revenue



  -For **Question 2 – Use GitHub Copilot in SSMS**, use the following SQL queries:

```sql id="q2sales1"
-- Find total sales per city

SELECT
    City,
    SUM(PurchaseAmount) AS TotalSales
FROM SalesData
GROUP BY City
ORDER BY TotalSales DESC;
```

```sql id="q2sales2"
-- Find Top 5 cities by revenue

SELECT TOP 5
    City,
    SUM(PurchaseAmount) AS TotalRevenue
FROM SalesData
GROUP BY City
ORDER BY TotalRevenue DESC;
```

### Expected Output:

* **Query 1:** Shows total sales aggregated for each city.
* **Query 2:** Returns only the **top 5 cities** with the highest revenue.

In **SSMS + GitHub Copilot**:

1. Open **New Query**.
2. Type the comment (example: `-- Find total sales per city`).
3. Let **Copilot generate the SQL**.
4. Press **Tab** to accept and run (**F5**).


Question 3  : Ask Copilot:

Find customers with purchases above average

  -For **Question 3 – Use GitHub Copilot in SSMS**

**Find customers with purchases above average**

```sql id="q3avgpurchase"
-- Find customers whose PurchaseAmount is greater than average purchase amount

SELECT
    CustomerID,
    Name,
    Age,
    City,
    PurchaseAmount,
    PurchaseDate
FROM SalesData
WHERE PurchaseAmount >
(
    SELECT AVG(PurchaseAmount)
    FROM SalesData
)
ORDER BY PurchaseAmount DESC;
```

### What this does:

* Calculates the **average PurchaseAmount** from `SalesData`
* Returns only customers whose **PurchaseAmount is greater than the average**
* Sorts results from **highest purchase to lowest**

**In SSMS + GitHub Copilot:**

1. Type the comment: `-- Find customers with purchases above average`
2. Wait for Copilot suggestion.
3. Press **Tab** to accept.
4. Run with **F5**.


Question 4 : Identify and handle missing values in the dataset using Copilot.

  -For **Question 4 – Identify and handle missing values in the dataset using GitHub Copilot (SQL in SSMS)**, use these queries.

### Step 1: Identify Missing Values

```sql id="q4identify"
-- Check missing (NULL) values in SalesData

SELECT
    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS Missing_Name,
    SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS Missing_Age,
    SUM(CASE WHEN City IS NULL THEN 1 ELSE 0 END) AS Missing_City,
    SUM(CASE WHEN PurchaseAmount IS NULL THEN 1 ELSE 0 END) AS Missing_PurchaseAmount,
    SUM(CASE WHEN PurchaseDate IS NULL THEN 1 ELSE 0 END) AS Missing_PurchaseDate
FROM SalesData;
```

### Step 2: Handle Missing Values

```sql id="q4handle"
-- Replace missing values

UPDATE SalesData
SET
    Name = ISNULL(Name, 'Unknown'),
    Age = ISNULL(Age, 30),
    City = ISNULL(City, 'Not Available'),
    PurchaseAmount = ISNULL(PurchaseAmount, 0),
    PurchaseDate = ISNULL(PurchaseDate, GETDATE());
```

### Step 3: Verify Missing Values Were Removed

```sql id="q4verify"
SELECT
    COUNT(*) AS RemainingMissing
FROM SalesData
WHERE
    Name IS NULL
    OR Age IS NULL
    OR City IS NULL
    OR PurchaseAmount IS NULL
    OR PurchaseDate IS NULL;
```

### Copilot workflow in SSMS:

1. Type comment: `-- Identify missing values in SalesData`
2. Accept Copilot suggestion (**Tab**).
3. Type: `-- Handle missing values`
4. Execute (**F5**).

This checks for **NULL values** and replaces them with default values.


 Question 5 : Task -

Identify duplicate rows

Remove duplicates using Copilot suggestion


 -For **Question 5 – Identify duplicate rows and remove duplicates using GitHub Copilot in SSMS**, use the following SQL.

### Step 1: Identify Duplicate Rows

This checks duplicate records based on all columns except `CustomerID` (since it is unique).

```sql id="q5identifydup"
-- Identify duplicate rows

SELECT
    Name,
    Age,
    City,
    PurchaseAmount,
    PurchaseDate,
    COUNT(*) AS DuplicateCount
FROM SalesData
GROUP BY
    Name,
    Age,
    City,
    PurchaseAmount,
    PurchaseDate
HAVING COUNT(*) > 1;
```

---

### Step 2: Remove Duplicate Rows

Keep the first occurrence and delete the rest.

```sql id="q5removedup"
-- Remove duplicate rows

WITH DuplicateRows AS
(
    SELECT
        CustomerID,
        ROW_NUMBER() OVER (
            PARTITION BY
                Name,
                Age,
                City,
                PurchaseAmount,
                PurchaseDate
            ORDER BY CustomerID
        ) AS RowNum
    FROM SalesData
)

DELETE FROM DuplicateRows
WHERE RowNum > 1;
```

---

### Step 3: Verify Duplicates Are Removed

```sql id="q5verify"
SELECT
    Name,
    Age,
    City,
    PurchaseAmount,
    PurchaseDate,
    COUNT(*) AS CountRows
FROM SalesData
GROUP BY
    Name,
    Age,
    City,
    PurchaseAmount,
    PurchaseDate
HAVING COUNT(*) > 1;
```

**Copilot in SSMS:**

1. Type comment: `-- Identify duplicate rows`
2. Accept Copilot suggestion (**Tab**)
3. Type: `-- Remove duplicates using ROW_NUMBER`
4. Run (**F5**)


uestion 6 : Generate:

Column chart OR bar chart and Interpret which region has highest sales

  -For **Question 6 – Generate Column Chart / Bar Chart and interpret which region (city) has highest sales**, use this SQL in **SSMS**.

### Step 1: Aggregate total sales by city

```sql id="q6chart"
-- Total sales by city

SELECT
    City AS Region,
    SUM(PurchaseAmount) AS TotalSales
FROM SalesData
GROUP BY City
ORDER BY TotalSales DESC;
```

### Step 2: Create Chart in SSMS

1. Run the query.
2. In the **Results** window → click **Chart** (or save results and visualize).
3. Select:

   * **Chart Type → Column Chart** *(or Bar Chart)*
   * **Category (X-axis) → Region**
   * **Value (Y-axis) → TotalSales**

### Interpretation

* The **tallest column / longest bar** represents the **region (city) with the highest sales**.
* Since the query is sorted in descending order, the **first row returned is the highest sales region**.

Optional query to display only the highest-sales region:

```sql id="q6topregion"
SELECT TOP 1
    City AS Region,
    SUM(PurchaseAmount) AS TotalSales
FROM SalesData
GROUP BY City
ORDER BY TotalSales DESC;
```

**Interpretation statement format:**
`<City Name> has the highest sales because it generated the maximum total PurchaseAmount among all regions.`


Question 7 : Create a pivot table showing total sales by region and product.


  -For **Question 7 – Create a Pivot Table showing total sales by region and product using SQL + GitHub Copilot in SSMS**.

> Assuming:
>
> * **Region = City**
> * **Product column exists in your dataset** (replace `Product` with your actual product column name if different)

```sql id="q7pivot"
-- Create Pivot Table: Total Sales by Region and Product

SELECT *
FROM
(
    SELECT
        City AS Region,
        Product,
        PurchaseAmount
    FROM SalesData
) AS SourceTable

PIVOT
(
    SUM(PurchaseAmount)
    FOR Product IN
    (
        [Product_A],
        [Product_B],
        [Product_C],
        [Product_D]
    )
) AS PivotTable

ORDER BY Region;
```

### What this does:

* Rows → **Region (City)**
* Columns → **Products**
* Values → **Total PurchaseAmount (Sales)**

### Example Output

| Region  | Product_A | Product_B | Product_C |
| ------- | --------: | --------: | --------: |
| Delhi   |     25000 |     18000 |     22000 |
| Mumbai  |     30000 |     21000 |     17000 |
| Chennai |     15000 |     25000 |     19000 |

### If product names are unknown, first check them:

```sql id="q7products"
SELECT DISTINCT Product
FROM SalesData;
```

### Copilot workflow in SSMS:

1. Type comment: `-- Create pivot table for total sales by region and product`
2. Accept Copilot suggestion (**Tab**)
3. Execute (**F5**)


Question 8 : Use Copilot to generate a new calculated column Total Sales.

  -For **Question 8 – Use GitHub Copilot to generate a new calculated column `TotalSales`** in SQL Server (SSMS).

If `TotalSales` should store the sales value for each row based on `PurchaseAmount`, create a **calculated/computed column**:

```sql id="q8totalsales"
-- Add a calculated column TotalSales

ALTER TABLE SalesData
ADD TotalSales AS (PurchaseAmount);
```

If instead you want a **physical column** and populate values:

```sql id="q8totalsalesupdate"
-- Create and populate TotalSales column

ALTER TABLE SalesData
ADD TotalSales DECIMAL(10,2);

UPDATE SalesData
SET TotalSales = PurchaseAmount;
```

### Verify the new column

```sql id="q8verify"
SELECT TOP 10
    CustomerID,
    Name,
    City,
    PurchaseAmount,
    TotalSales
FROM SalesData;
```

### Copilot workflow in SSMS:

1. Type comment: `-- Create calculated column TotalSales`
2. Accept Copilot suggestion (**Tab**)
3. Run (**F5**)

This creates a **new TotalSales column** derived from the existing sales amount.


 Question 9: Summarize the dataset and find the key insights

   -For **Question 9 – Summarize the dataset and find key insights using GitHub Copilot in SSMS**, use aggregate SQL queries.

### Step 1: Generate Dataset Summary

```sql id="q9summary"
-- Dataset Summary

SELECT
    COUNT(*) AS TotalRecords,
    COUNT(DISTINCT CustomerID) AS TotalCustomers,
    AVG(Age) AS AverageAge,
    SUM(PurchaseAmount) AS TotalSales,
    AVG(PurchaseAmount) AS AveragePurchase,
    MIN(PurchaseAmount) AS MinimumPurchase,
    MAX(PurchaseAmount) AS MaximumPurchase
FROM SalesData;
```

---

### Step 2: Sales by Region (City)

```sql id="q9city"
-- Total sales by region

SELECT
    City,
    SUM(PurchaseAmount) AS TotalSales
FROM SalesData
GROUP BY City
ORDER BY TotalSales DESC;
```

---

### Step 3: Top Customers

```sql id="q9customer"
-- Top customers by spending

SELECT TOP 10
    CustomerID,
    Name,
    City,
    SUM(PurchaseAmount) AS TotalSpent
FROM SalesData
GROUP BY CustomerID, Name, City
ORDER BY TotalSpent DESC;
```

---

### Step 4: Purchase Trend

```sql id="q9trend"
-- Monthly sales trend

SELECT
    YEAR(PurchaseDate) AS Year,
    MONTH(PurchaseDate) AS Month,
    SUM(PurchaseAmount) AS MonthlySales
FROM SalesData
GROUP BY
    YEAR(PurchaseDate),
    MONTH(PurchaseDate)
ORDER BY Year, Month;
```

### Key Insights (Interpretation Template)

* **Total Records:** Number of customer purchase records.
* **Total Sales:** Overall revenue generated.
* **Highest Sales Region:** City with maximum total purchase amount.
* **Average Customer Spend:** Mean purchase amount per transaction.
* **Top Customers:** Customers contributing the most revenue.
* **Sales Trend:** Observe peak and low-performing months.

**Copilot in SSMS:**

1. Type comments such as `-- Summarize dataset`
2. Accept Copilot suggestions (**Tab**)
3. Execute (**F5**) and interpret results.


For **Question 9 – Summarize the dataset and find key insights using GitHub Copilot in SSMS**, use aggregate SQL queries.

### Step 1: Generate Dataset Summary

```sql id="q9summary"
-- Dataset Summary

SELECT
    COUNT(*) AS TotalRecords,
    COUNT(DISTINCT CustomerID) AS TotalCustomers,
    AVG(Age) AS AverageAge,
    SUM(PurchaseAmount) AS TotalSales,
    AVG(PurchaseAmount) AS AveragePurchase,
    MIN(PurchaseAmount) AS MinimumPurchase,
    MAX(PurchaseAmount) AS MaximumPurchase
FROM SalesData;
```

---

### Step 2: Sales by Region (City)

```sql id="q9city"
-- Total sales by region

SELECT
    City,
    SUM(PurchaseAmount) AS TotalSales
FROM SalesData
GROUP BY City
ORDER BY TotalSales DESC;
```

---

### Step 3: Top Customers

```sql id="q9customer"
-- Top customers by spending

SELECT TOP 10
    CustomerID,
    Name,
    City,
    SUM(PurchaseAmount) AS TotalSpent
FROM SalesData
GROUP BY CustomerID, Name, City
ORDER BY TotalSpent DESC;
```

---

### Step 4: Purchase Trend

```sql id="q9trend"
-- Monthly sales trend

SELECT
    YEAR(PurchaseDate) AS Year,
    MONTH(PurchaseDate) AS Month,
    SUM(PurchaseAmount) AS MonthlySales
FROM SalesData
GROUP BY
    YEAR(PurchaseDate),
    MONTH(PurchaseDate)
ORDER BY Year, Month;
```

### Key Insights (Interpretation Template)

* **Total Records:** Number of customer purchase records.
* **Total Sales:** Overall revenue generated.
* **Highest Sales Region:** City with maximum total purchase amount.
* **Average Customer Spend:** Mean purchase amount per transaction.
* **Top Customers:** Customers contributing the most revenue.
* **Sales Trend:** Observe peak and low-performing months.

**Copilot in SSMS:**

1. Type comments such as `-- Summarize dataset`
2. Accept Copilot suggestions (**Tab**)
3. Execute (**F5**) and interpret results.


 Question 10 : Generate VBA code using Copilot for cleaning data (removing duplicates, filling null values,
format table).


  -For **Question 10 – Generate VBA code using GitHub Copilot for cleaning data (remove duplicates, fill null values, format table)**, use this VBA macro in **Excel VBA Editor (Alt + F11)**.

```vb
Sub CleanSalesData()

    Dim ws As Worksheet
    Dim lastRow As Long
    Dim lastCol As Long
    Dim rng As Range

    Set ws = ActiveSheet

    ' Find last used row and column
    lastRow = ws.Cells(ws.Rows.Count, 1).End(xlUp).Row
    lastCol = ws.Cells(1, ws.Columns.Count).End(xlToLeft).Column

    Set rng = ws.Range(ws.Cells(1, 1), ws.Cells(lastRow, lastCol))

    ' Remove duplicates
    rng.RemoveDuplicates Columns:=Array(1, 2, 3, 4, 5, 6), Header:=xlYes

    ' Refresh last row after removing duplicates
    lastRow = ws.Cells(ws.Rows.Count, 1).End(xlUp).Row

    ' Fill blank values
    Dim c As Range

    For Each c In ws.Range(ws.Cells(2, 1), ws.Cells(lastRow, lastCol))

        If IsEmpty(c.Value) Then

            Select Case c.Column
                Case 2
                    c.Value = "Unknown"

                Case 3
                    c.Value = 0

                Case 4
                    c.Value = "Not Available"

                Case 5
                    c.Value = 0

                Case 6
                    c.Value = Date

            End Select

        End If

    Next c

    ' Format as table
    Dim tbl As ListObject

    On Error Resume Next
    ws.ListObjects(1).Delete
    On Error GoTo 0

    Set tbl = ws.ListObjects.Add( _
        SourceType:=xlSrcRange, _
        Source:=ws.Range(ws.Cells(1, 1), ws.Cells(lastRow, lastCol)), _
        XlListObjectHasHeaders:=xlYes)

    tbl.Name = "SalesDataTable"
    tbl.TableStyle = "TableStyleMedium2"

    ' Auto fit columns
    ws.Cells.EntireColumn.AutoFit

    MsgBox "Data cleaning completed successfully!"

End Sub
```

### Steps to run in Excel

1. Open Excel → **Alt + F11**
2. **Insert → Module**
3. Paste the VBA code
4. Press **F5** to run

### This macro performs:

✔ Removes duplicate rows
✔ Fills blank/null values
✔ Converts data into a formatted Excel table
✔ Auto-adjusts column widths

